## 1. Imports

In [1]:
import os
import gc
import warnings

import numpy as np
import pandas as pd

import networkx as nx

from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics.pairwise import cosine_similarity

from tqdm.auto import tqdm

warnings.filterwarnings("ignore")

tqdm.pandas()

## 2. Paths

In [2]:
PROJECT_PATH = "/content/drive/MyDrive/FinSight_AI"

DATA_PATH = os.path.join(
    PROJECT_PATH,
    "data"
)

PROCESSED_PATH = os.path.join(
    DATA_PATH,
    "processed"
)

EVENT_PATH = os.path.join(
    PROCESSED_PATH,
    "event_propagation"
)

os.makedirs(
    EVENT_PATH,
    exist_ok=True
)

## 3. Load Datasets

In [3]:
FINANCIAL_DATASET = os.path.join(
    PROCESSED_PATH,
    "financial_intelligence",
    "financial_intelligence_dataset.parquet"
)

COMPANY_ANALYTICS = os.path.join(
    PROCESSED_PATH,
    "company_analytics",
    "company_analytics.parquet"
)

TIMELINE_DATA = os.path.join(
    PROCESSED_PATH,
    "company_timeline",
    "company_event_timeline.parquet"
)

ENTITY_DATA = os.path.join(
    PROCESSED_PATH,
    "entities",
    "news_entities.parquet"
)

financial_df = pd.read_parquet(
    FINANCIAL_DATASET,
    columns=[
        "news_id",
        "ticker",
        "publisher",
        "published_date",
        "final_event",
        "market_signal",
        "final_confidence"
    ]
)

analytics_df = pd.read_parquet(
    COMPANY_ANALYTICS
)

timeline_df = pd.read_parquet(
    TIMELINE_DATA
)

entity_df = pd.read_parquet(
    ENTITY_DATA,
    columns=[
        "news_id",
        "entity",
        "entity_label"
    ]
)

print("Financial :", financial_df.shape)
print("Analytics :", analytics_df.shape)
print("Timeline :", timeline_df.shape)
print("Entities :", entity_df.shape)

Financial : (3215296, 7)
Analytics : (6619, 70)
Timeline : (3215296, 26)
Entities : (7345743, 3)


## 4. Validation

In [4]:
print()

print("Companies :", financial_df["ticker"].nunique())

print("Publishers :", financial_df["publisher"].nunique())

print("Events :", financial_df["final_event"].nunique())

print("Market Signals :", financial_df["market_signal"].nunique())

print("Entity Types :", entity_df["entity_label"].nunique())


Companies : 6619
Publishers : 1047
Events : 30
Market Signals : 4
Entity Types : 21


## 5. Event Statistics

In [5]:
event_statistics = (

    financial_df

    .groupby(

        "final_event"

    )

    .agg(

        total_news=("news_id","count"),

        companies=("ticker","nunique"),

        publishers=("publisher","nunique"),

        avg_confidence=("final_confidence","mean"),

        bullish=(

            "market_signal",

            lambda x:(x=="Bullish").sum()

        ),

        bearish=(

            "market_signal",

            lambda x:(x=="Bearish").sum()

        ),

        neutral=(

            "market_signal",

            lambda x:(x=="Neutral").sum()

        )

    )

    .reset_index()

)

display(event_statistics)

,final_event,total_news,companies,publishers,avg_confidence,bullish,bearish,neutral
0,Analyst Rating,277248,5178,311,0.899277,78310,12691,184055
1,Bankruptcy,3043,1142,93,0.904568,37,1826,1062
2,Commodity,176547,4965,450,0.840405,9460,7778,157888
3,Credit Rating,36358,3955,208,0.956756,2697,1122,32488
4,Cryptocurrency,13739,2556,187,0.773372,119,74,13543
5,Cybersecurity,7612,1817,171,0.803114,155,928,6485
6,Dividend,110494,3981,172,0.956925,107799,39,1144
7,ESG,34695,4120,250,0.746687,118,26,34550
8,Earnings,567435,5694,396,0.944125,109435,45312,390842
9,Economic Data,14319,2222,172,0.859742,0,0,14319


## 6. Event Timeline Statistics

In [6]:
financial_df["event_date"] = pd.to_datetime(

    financial_df["published_date"]

).dt.date

event_timeline = (

    financial_df

    .groupby(

        [

            "event_date",

            "final_event"

        ]

    )

    .size()

    .reset_index(

        name="news_count"

    )

)

display(event_timeline.head())

,event_date,final_event,news_count
0,2009-02-14,Cryptocurrency,1
1,2009-04-27,Product Launch,2
2,2009-04-29,Product Launch,1
3,2009-05-22,Product Launch,1
4,2009-05-27,Market Movement,3


## 7. Event Diversity

In [7]:
event_diversity = (

    financial_df

    .groupby(

        "final_event"

    )

    .agg(

        companies=("ticker","nunique"),

        publishers=("publisher","nunique"),

        signals=("market_signal","nunique")

    )

    .reset_index()

)

display(event_diversity)

,final_event,companies,publishers,signals
0,Analyst Rating,5178,311,4
1,Bankruptcy,1142,93,4
2,Commodity,4965,450,4
3,Credit Rating,3955,208,4
4,Cryptocurrency,2556,187,4
5,Cybersecurity,1817,171,4
6,Dividend,3981,172,4
7,ESG,4120,250,4
8,Earnings,5694,396,4
9,Economic Data,2222,172,1


## 8. Merge Statistics

In [8]:
event_statistics = event_statistics.merge(

    event_diversity,

    on="final_event",

    suffixes=(

        "",

        "_check"

    )

)

event_statistics = event_statistics.loc[

    :,

    ~event_statistics.columns.str.endswith(

        "_check"

    )

]

display(event_statistics.head())

,final_event,total_news,companies,publishers,avg_confidence,bullish,bearish,neutral,signals
0,Analyst Rating,277248,5178,311,0.899277,78310,12691,184055,4
1,Bankruptcy,3043,1142,93,0.904568,37,1826,1062,4
2,Commodity,176547,4965,450,0.840405,9460,7778,157888,4
3,Credit Rating,36358,3955,208,0.956756,2697,1122,32488,4
4,Cryptocurrency,13739,2556,187,0.773372,119,74,13543,4


## 9. Save

In [9]:
event_statistics.to_parquet(

    os.path.join(

        EVENT_PATH,

        "event_statistics.parquet"

    ),

    index=False

)

event_timeline.to_parquet(

    os.path.join(

        EVENT_PATH,

        "event_timeline.parquet"

    ),

    index=False

)

gc.collect()

print("Saved.")

Saved.


## 10. Event -> Company Graph

In [10]:
event_company_graph = (

    financial_df

    .groupby(

        [

            "final_event",

            "ticker"

        ],

        observed=True

    )

    .agg(

        news_count=("news_id", "count"),

        avg_confidence=("final_confidence", "mean")

    )

    .reset_index()

)

event_company_graph["source"] = (

    "EVENT_"

    + event_company_graph["final_event"]

        .str.replace(" ", "_", regex=False)

)

event_company_graph["target"] = (

    "COMP_"

    + event_company_graph["ticker"].astype(str)

)

event_company_graph["relation"] = "AFFECTS"

event_company_graph = event_company_graph[

    [

        "source",

        "target",

        "relation",

        "news_count",

        "avg_confidence"

    ]

]

print(event_company_graph.shape)

display(event_company_graph.head())

(116520, 5)


,source,target,relation,news_count,avg_confidence
0,EVENT_Analyst_Rating,COMP_A,AFFECTS,207,0.904322
1,EVENT_Analyst_Rating,COMP_AA,AFFECTS,301,0.948551
2,EVENT_Analyst_Rating,COMP_AAC,AFFECTS,18,0.865650
3,EVENT_Analyst_Rating,COMP_AAL,AFFECTS,114,0.920354
4,EVENT_Analyst_Rating,COMP_AAMC,AFFECTS,2,0.851250


## 11. Save Company Graph

In [12]:
event_company_graph.to_parquet(

    os.path.join(

        EVENT_PATH,

        "event_company_graph.parquet"

    ),

    index=False

)

print("Saved Event -> Company Graph")

Saved Event -> Company Graph


## 12. Event -> Publisher Graph

In [13]:
event_publisher_graph = (

    financial_df

    .groupby(

        [

            "final_event",

            "publisher"

        ],

        observed=True

    )

    .agg(

        news_count=("news_id", "count"),

        avg_confidence=("final_confidence", "mean")

    )

    .reset_index()

)

event_publisher_graph["source"] = (

    "EVENT_"

    + event_publisher_graph["final_event"]

        .str.replace(" ", "_", regex=False)

)

event_publisher_graph["target"] = (

    "PUB_"

    + event_publisher_graph["publisher"]

        .astype(str)

        .str.replace(" ", "_", regex=False)

)

event_publisher_graph["relation"] = "REPORTED_BY"

event_publisher_graph = event_publisher_graph[

    [

        "source",

        "target",

        "relation",

        "news_count",

        "avg_confidence"

    ]

]

print(event_publisher_graph.shape)

display(event_publisher_graph.head())

(7479, 5)


,source,target,relation,news_count,avg_confidence
0,EVENT_Analyst_Rating,PUB_ABNNewswire,REPORTED_BY,2,1.000000
1,EVENT_Analyst_Rating,PUB_Aaron_Levitt,REPORTED_BY,4,0.787300
2,EVENT_Analyst_Rating,PUB_Abe_Raymond,REPORTED_BY,16,0.827619
3,EVENT_Analyst_Rating,PUB_Accesswire,REPORTED_BY,211,0.829685
4,EVENT_Analyst_Rating,PUB_Adam_Parker,REPORTED_BY,6,0.833333


## 13. Save Publisher Graph

In [15]:
event_publisher_graph.to_parquet(

    os.path.join(

        EVENT_PATH,

        "event_publisher_graph.parquet"

    ),

    index=False

)

print("Saved Event -> Publisher Graph")

Saved Event -> Publisher Graph


## 14. Event -> Entity Graph

In [16]:
event_entity = (

    financial_df

    [

        [

            "news_id",

            "final_event"

        ]

    ]

    .merge(

        entity_df,

        on="news_id",

        how="inner"

    )

)

In [17]:
print(event_entity.shape)

display(event_entity.head())

(7345743, 4)


,news_id,final_event,entity,entity_label
0,3,Market Movement,Biggest Movers,ORG
1,5,Analyst Rating,Price Target,FINANCIAL_METRIC
2,5,Analyst Rating,Maintains,ANALYST_ACTION
3,5,Analyst Rating,Neutral,ANALYST_ACTION
4,5,Analyst Rating,Price Target to $88,PRICE_TARGET


## 15. Build Event -> Entity Graph

In [18]:
event_entity_graph = (

    event_entity

    .groupby(

        [

            "final_event",

            "entity",

            "entity_label"

        ],

        observed=True

    )

    .size()

    .reset_index(name="mentions")

)

event_entity_graph["source"] = (

    "EVENT_"

    + event_entity_graph["final_event"]

        .str.replace(" ", "_", regex=False)

)

event_entity_graph["target"] = (

    "ENTITY_"

    + event_entity_graph["entity"]

        .astype(str)

        .str.replace(" ", "_", regex=False)

)

event_entity_graph["relation"] = "MENTIONS"

event_entity_graph = event_entity_graph[

    [

        "source",

        "target",

        "relation",

        "mentions",

        "entity_label"

    ]

]

print(event_entity_graph.shape)

display(event_entity_graph.head())

(1198728, 5)


,source,target,relation,mentions,entity_label
0,EVENT_Analyst_Rating,ENTITY_#1_Customer,MENTIONS,1,MONEY
1,EVENT_Analyst_Rating,ENTITY_#1_Priority,MENTIONS,1,MONEY
2,EVENT_Analyst_Rating,ENTITY_#1_Rank,MENTIONS,4,MONEY
3,EVENT_Analyst_Rating,ENTITY_$_32,MENTIONS,1,MONEY
4,EVENT_Analyst_Rating,ENTITY_$_7,MENTIONS,1,MONEY


## 16. Save Entity Graph

In [20]:
event_entity_graph.to_parquet(

    os.path.join(

        EVENT_PATH,

        "event_entity_graph.parquet"

    ),

    index=False

)

print("Saved Event -> Entity Graph")

Saved Event -> Entity Graph


## 17. Free Memory

In [21]:
del event_entity

gc.collect()

print("Memory Cleared")

Memory Cleared


## 18. Event Influence Score

In [22]:
event_statistics["timeline_days"] = (

    event_timeline

    .groupby("final_event")["event_date"]

    .nunique()

    .reindex(event_statistics["final_event"])

    .values

)

scaler = MinMaxScaler()

scaled = scaler.fit_transform(

    event_statistics[

        [

            "companies",

            "publishers",

            "avg_confidence",

            "timeline_days"

        ]

    ]

)

scaled = pd.DataFrame(

    scaled,

    columns=[

        "companies_score",

        "publishers_score",

        "confidence_score",

        "timeline_score"

    ]

)

event_statistics = pd.concat(

    [

        event_statistics,

        scaled

    ],

    axis=1

)

event_statistics["influence_score"] = (

      0.35 * event_statistics["companies_score"]

    + 0.25 * event_statistics["publishers_score"]

    + 0.25 * event_statistics["confidence_score"]

    + 0.15 * event_statistics["timeline_score"]

)

event_statistics["influence_score"] *= 100

event_statistics["influence_score"] = (

    event_statistics["influence_score"]

    .round(2)

)

display(

    event_statistics[

        [

            "final_event",

            "influence_score"

        ]

    ]

    .sort_values(

        "influence_score",

        ascending=False

    )

)

,final_event,influence_score
23,Product Launch,82.23
17,Market Movement,78.84
8,Earnings,78.41
2,Commodity,66.03
0,Analyst Rating,65.78
28,Technical Analysis,65.30
18,Merger & Acquisition,64.38
6,Dividend,59.04
10,Executive Change,58.01
3,Credit Rating,57.10


## 19. Event Impact Category

In [23]:
event_statistics["impact_category"] = pd.qcut(

    event_statistics["influence_score"],

    q=4,

    labels=[

        "Low",

        "Medium",

        "High",

        "Very High"

    ]

)

display(

    event_statistics[

        [

            "final_event",

            "influence_score",

            "impact_category"

        ]

    ]

    .sort_values(

        "influence_score",

        ascending=False

    )

)

,final_event,influence_score,impact_category
23,Product Launch,82.23,Very High
17,Market Movement,78.84,Very High
8,Earnings,78.41,Very High
2,Commodity,66.03,Very High
0,Analyst Rating,65.78,Very High
28,Technical Analysis,65.30,Very High
18,Merger & Acquisition,64.38,Very High
6,Dividend,59.04,Very High
10,Executive Change,58.01,High
3,Credit Rating,57.10,High


## 20. Save Influence Dataset

In [24]:
event_statistics.to_parquet(

    os.path.join(

        EVENT_PATH,

        "event_influence.parquet"

    ),

    index=False

)

print("Saved Event Influence")

Saved Event Influence


## 21. Company Influence Score

In [25]:
company_influence = (

    financial_df

    .groupby("ticker")

    .agg(

        total_news=("news_id","count"),

        event_types=("final_event","nunique"),

        publishers=("publisher","nunique"),

        avg_confidence=("final_confidence","mean")

    )

    .reset_index()

)

## 22. Normalize Company Influence

In [26]:
scaled = MinMaxScaler().fit_transform(

    company_influence[

        [

            "total_news",

            "event_types",

            "publishers",

            "avg_confidence"

        ]

    ]

)

scaled = pd.DataFrame(

    scaled,

    columns=[

        "news_score",

        "event_score",

        "publisher_score",

        "confidence_score"

    ]

)

company_influence = pd.concat(

    [

        company_influence,

        scaled

    ],

    axis=1

)

company_influence["company_influence_score"] = (

      0.35 * company_influence["news_score"]

    + 0.30 * company_influence["event_score"]

    + 0.20 * company_influence["publisher_score"]

    + 0.15 * company_influence["confidence_score"]

)

company_influence["company_influence_score"] *= 100

company_influence["company_influence_score"] = (

    company_influence["company_influence_score"]

    .round(2)

)

display(

    company_influence

    .sort_values(

        "company_influence_score",

        ascending=False

    )

    .head(20)

)

,ticker,total_news,event_types,publishers,avg_confidence,news_score,event_score,publisher_score,confidence_score,company_influence_score
2746,HD,4415,30,232,0.829553,0.872332,1.000000,0.927711,0.659106,88.97
3486,KR,5061,30,168,0.827025,1.000000,1.000000,0.670683,0.654051,88.22
2523,GILD,4938,30,169,0.820607,0.975692,1.000000,0.674699,0.641213,87.26
2159,FDX,4358,30,209,0.831489,0.861067,1.000000,0.835341,0.662979,86.79
1555,DISH,4544,30,153,0.823487,0.897826,1.000000,0.610442,0.646974,83.34
3386,JWN,4361,30,162,0.823712,0.861660,1.000000,0.646586,0.647424,82.80
3778,MDT,4421,30,134,0.851852,0.873518,1.000000,0.534137,0.703704,81.81
2144,FCX,4253,30,160,0.816144,0.840316,1.000000,0.638554,0.632288,81.67
1711,EA,3909,30,176,0.829336,0.772332,1.000000,0.702811,0.658672,80.97
2633,GRPN,4020,30,165,0.828431,0.794269,1.000000,0.658635,0.656862,80.83


## 23. Save Company Influence

In [27]:
company_influence.to_parquet(

    os.path.join(

        EVENT_PATH,

        "company_influence.parquet"

    ),

    index=False

)

print("Saved Company Influence")

Saved Company Influence


## 24. Event Fingerprints

In [28]:
def build_event_fingerprint(row):

    fp = []

    if row["impact_category"] == "Very High":
        fp.append("High Impact")

    if row["avg_confidence"] > 0.90:
        fp.append("Reliable")

    if row["companies"] > event_statistics["companies"].median():
        fp.append("Broad Coverage")

    if row["publishers"] > event_statistics["publishers"].median():
        fp.append("Widely Reported")

    return ", ".join(fp)

event_statistics["event_fingerprint"] = (

    event_statistics.apply(

        build_event_fingerprint,

        axis=1

    )

)

display(

    event_statistics[

        [

            "final_event",

            "event_fingerprint"

        ]

    ]

)

,final_event,event_fingerprint
0,Analyst Rating,"High Impact, Broad Coverage, Widely Reported"
1,Bankruptcy,Reliable
2,Commodity,"High Impact, Broad Coverage, Widely Reported"
3,Credit Rating,"Reliable, Widely Reported"
4,Cryptocurrency,
5,Cybersecurity,
6,Dividend,"High Impact, Reliable, Broad Coverage"
7,ESG,"Broad Coverage, Widely Reported"
8,Earnings,"High Impact, Reliable, Broad Coverage, Widely ..."
9,Economic Data,


## 25. Explainability

In [29]:
event_statistics["explanation"] = (

    "Companies: "

    + event_statistics["companies"].astype(str)

    + " | Publishers: "

    + event_statistics["publishers"].astype(str)

    + " | Confidence: "

    + event_statistics["avg_confidence"]

        .round(2)

        .astype(str)

    + " | Timeline Days: "

    + event_statistics["timeline_days"].astype(str)

)

display(

    event_statistics[

        [

            "final_event",

            "explanation"

        ]

    ]

)

,final_event,explanation
0,Analyst Rating,Companies: 5178 | Publishers: 311 | Confidence...
1,Bankruptcy,Companies: 1142 | Publishers: 93 | Confidence:...
2,Commodity,Companies: 4965 | Publishers: 450 | Confidence...
3,Credit Rating,Companies: 3955 | Publishers: 208 | Confidence...
4,Cryptocurrency,Companies: 2556 | Publishers: 187 | Confidence...
5,Cybersecurity,Companies: 1817 | Publishers: 171 | Confidence...
6,Dividend,Companies: 3981 | Publishers: 172 | Confidence...
7,ESG,Companies: 4120 | Publishers: 250 | Confidence...
8,Earnings,Companies: 5694 | Publishers: 396 | Confidence...
9,Economic Data,Companies: 2222 | Publishers: 172 | Confidence...


## 26. Save Updated Statistics

In [30]:
event_statistics.to_parquet(

    os.path.join(

        EVENT_PATH,

        "event_statistics.parquet"

    ),

    index=False

)

gc.collect()

print("Updated Event Statistics Saved")

Updated Event Statistics Saved


## 27. Build Event Graph (NetworkX)

In [37]:
event_company = (

    financial_df

    .groupby(

        [

            "final_event",

            "ticker"

        ]

    )

    .size()

    .reset_index(name="count")

)

In [47]:
event_matrix = (

    event_company

    .pivot_table(

        index="ticker",

        columns="final_event",

        values="count",

        fill_value=0

    )

)

In [48]:
from sklearn.metrics.pairwise import cosine_similarity

similarity = cosine_similarity(

    event_matrix.T

)

similarity = pd.DataFrame(

    similarity,

    index=event_matrix.columns,

    columns=event_matrix.columns

)

In [49]:
edges = []

threshold = 0.35

for i in similarity.index:

    for j in similarity.columns:

        if i >= j:
            continue

        score = similarity.loc[i, j]

        if score >= threshold:

            edges.append(

                [

                    i,

                    j,

                    score

                ]

            )

event_edge_df = pd.DataFrame(

    edges,

    columns=[

        "event_a",

        "event_b",

        "weight"

    ]

)

## 28. Build NetworkX Graph

In [50]:
G = nx.Graph()

for row in event_edge_df.itertuples(index=False):

    G.add_edge(

        row.event_a,

        row.event_b,

        weight=row.weight

    )

print("Nodes :", G.number_of_nodes())

print("Edges :", G.number_of_edges())

Nodes : 29
Edges : 221


In [58]:
all_events = sorted(
    financial_df["final_event"].unique()
)

for event in all_events:

    if event not in G:

        G.add_node(event)

print("Nodes :", G.number_of_nodes())
print("Edges :", G.number_of_edges())

Nodes : 30
Edges : 221


In [59]:
all_events = set(financial_df["final_event"].unique())

graph_events = set(G.nodes())

missing = all_events - graph_events

print("Missing Events:")
print(missing)

Missing Events:
set()


## 29. Degree Centrality

In [60]:
degree = nx.degree_centrality(G)

degree = (

    pd.DataFrame(

        degree.items(),

        columns=[

            "final_event",

            "degree_centrality"

        ]

    )

)

display(

    degree.sort_values(

        "degree_centrality",

        ascending=False

    )

)

,final_event,degree_centrality
17,Product Launch,0.896552
12,Market Movement,0.827586
18,Revenue Guidance,0.827586
21,Technical Analysis,0.793103
10,Layoffs,0.793103
0,Analyst Rating,0.758621
15,Other,0.758621
4,Earnings,0.758621
5,Executive Change,0.724138
13,Merger & Acquisition,0.724138


In [61]:
event_centrality["isolated"] = (
    event_centrality["degree_centrality"] == 0
)

event_centrality["graph_status"] = np.where(
    event_centrality["isolated"],
    "Isolated Event",
    "Connected Event"
)

display(event_centrality)

,final_event,degree_centrality,betweenness,closeness,eigenvector,isolated,graph_status
0,Analyst Rating,0.785714,0.026455,0.823529,0.266636,False,Connected Event
1,Credit Rating,0.464286,0.000000,0.651163,0.133294,False,Connected Event
2,Dividend,0.428571,0.000000,0.636364,0.129534,False,Connected Event
3,ESG,0.178571,0.021164,0.538462,0.051468,False,Connected Event
4,Earnings,0.785714,0.007937,0.823529,0.268049,False,Connected Event
5,Executive Change,0.750000,0.000000,0.800000,0.248416,False,Connected Event
6,Fraud,0.428571,0.000000,0.636364,0.122930,False,Connected Event
7,Funding,0.571429,0.005291,0.700000,0.170663,False,Connected Event
8,IPO,0.571429,0.002646,0.700000,0.163523,False,Connected Event
9,Insider Trading,0.250000,0.000000,0.549020,0.077390,False,Connected Event


## 30. Betweenness Centrality

In [62]:
betweenness = nx.betweenness_centrality(

    G,

    weight="weight"

)

betweenness = (

    pd.DataFrame(

        betweenness.items(),

        columns=[

            "final_event",

            "betweenness"

        ]

    )

)

display(

    betweenness.sort_values(

        "betweenness",

        ascending=False

    )

)

,final_event,betweenness
10,Layoffs,0.088670
18,Revenue Guidance,0.076355
14,Options Activity,0.039409
15,Other,0.036946
22,Trading Halt,0.036946
21,Technical Analysis,0.034483
17,Product Launch,0.034483
16,Partnership,0.024631
0,Analyst Rating,0.024631
11,Litigation,0.019704


## 31. Closeness Centrality

In [63]:
closeness = nx.closeness_centrality(G)

closeness = (

    pd.DataFrame(

        closeness.items(),

        columns=[

            "final_event",

            "closeness"

        ]

    )

)

display(

    closeness.sort_values(

        "closeness",

        ascending=False

    )

)

,final_event,closeness
17,Product Launch,0.901149
12,Market Movement,0.844828
18,Revenue Guidance,0.844828
21,Technical Analysis,0.819227
10,Layoffs,0.819227
0,Analyst Rating,0.795132
15,Other,0.795132
4,Earnings,0.795132
5,Executive Change,0.772414
13,Merger & Acquisition,0.772414


## 32. Eigenvector Centrality

In [64]:
eigen = nx.eigenvector_centrality(

    G,

    max_iter=1000,

    weight="weight"

)

eigen = (

    pd.DataFrame(

        eigen.items(),

        columns=[

            "final_event",

            "eigenvector"

        ]

    )

)

display(

    eigen.sort_values(

        "eigenvector",

        ascending=False

    )

)

,final_event,eigenvector
17,Product Launch,2.867286e-01
12,Market Movement,2.813059e-01
4,Earnings,2.680494e-01
0,Analyst Rating,2.666363e-01
18,Revenue Guidance,2.536433e-01
13,Merger & Acquisition,2.494183e-01
5,Executive Change,2.484163e-01
21,Technical Analysis,2.476549e-01
10,Layoffs,2.321773e-01
14,Options Activity,2.289272e-01


## 33. Merge All Centralities

In [65]:
event_centrality = (

    degree

    .merge(

        betweenness,

        on="final_event"

    )

    .merge(

        closeness,

        on="final_event"

    )

    .merge(

        eigen,

        on="final_event"

    )

)

display(event_centrality)

,final_event,degree_centrality,betweenness,closeness,eigenvector
0,Analyst Rating,0.758621,0.024631,0.795132,2.666363e-01
1,Credit Rating,0.448276,0.000000,0.628709,1.332943e-01
2,Dividend,0.413793,0.000000,0.614420,1.295340e-01
3,ESG,0.172414,0.019704,0.519894,5.146788e-02
4,Earnings,0.758621,0.007389,0.795132,2.680494e-01
5,Executive Change,0.724138,0.000000,0.772414,2.484163e-01
6,Fraud,0.413793,0.000000,0.614420,1.229300e-01
7,Funding,0.551724,0.004926,0.675862,1.706629e-01
8,IPO,0.551724,0.002463,0.675862,1.635231e-01
9,Insider Trading,0.241379,0.000000,0.530088,7.739037e-02


## 34. Save

In [66]:
event_centrality.to_parquet(

    os.path.join(

        EVENT_PATH,

        "event_centrality.parquet"

    ),

    index=False

)

print("Saved Event Centrality")

Saved Event Centrality


## 35. Graph Importance Score

In [67]:
from sklearn.preprocessing import MinMaxScaler

graph_scores = event_centrality[
    [
        "degree_centrality",
        "betweenness",
        "closeness",
        "eigenvector"
    ]
]

scaled = MinMaxScaler().fit_transform(graph_scores)

scaled = pd.DataFrame(

    scaled,

    columns=[

        "degree_scaled",

        "betweenness_scaled",

        "closeness_scaled",

        "eigenvector_scaled"

    ]

)

event_centrality = pd.concat(

    [

        event_centrality,

        scaled

    ],

    axis=1

)

event_centrality["graph_importance"] = (

      0.30 * event_centrality["degree_scaled"]
    + 0.20 * event_centrality["betweenness_scaled"]
    + 0.20 * event_centrality["closeness_scaled"]
    + 0.30 * event_centrality["eigenvector_scaled"]

) * 100

event_centrality["graph_importance"] = (
    event_centrality["graph_importance"]
    .round(2)
)

display(

    event_centrality

    .sort_values(

        "graph_importance",

        ascending=False

    )
)

,final_event,degree_centrality,betweenness,closeness,eigenvector,degree_scaled,betweenness_scaled,closeness_scaled,eigenvector_scaled,graph_importance
18,Revenue Guidance,0.827586,0.076355,0.844828,2.536433e-01,0.923077,0.861111,0.937500,0.884611,90.20
10,Layoffs,0.793103,0.088670,0.819227,2.321773e-01,0.884615,1.000000,0.909091,0.809746,89.01
17,Product Launch,0.896552,0.034483,0.901149,2.867286e-01,1.000000,0.388889,1.000000,1.000000,87.78
12,Market Movement,0.827586,0.017241,0.844828,2.813059e-01,0.923077,0.194444,0.937500,0.981088,79.76
21,Technical Analysis,0.793103,0.034483,0.819227,2.476549e-01,0.884615,0.388889,0.909091,0.863726,78.41
0,Analyst Rating,0.758621,0.024631,0.795132,2.666363e-01,0.846154,0.277778,0.882353,0.929926,76.49
15,Other,0.758621,0.036946,0.795132,2.212083e-01,0.846154,0.416667,0.882353,0.771490,74.51
4,Earnings,0.758621,0.007389,0.795132,2.680494e-01,0.846154,0.083333,0.882353,0.934854,72.74
14,Options Activity,0.689655,0.039409,0.750958,2.289272e-01,0.769231,0.444444,0.833333,0.798411,72.58
13,Merger & Acquisition,0.724138,0.000000,0.772414,2.494183e-01,0.807692,0.000000,0.857143,0.869876,67.47


## 36. Event Connectivity Level

In [68]:
event_centrality["connectivity_level"] = pd.cut(

    event_centrality["graph_importance"],

    bins=[

        -1,
        20,
        40,
        60,
        80,
        100

    ],

    labels=[

        "Very Low",

        "Low",

        "Medium",

        "High",

        "Very High"

    ]

)

display(

    event_centrality[

        [

            "final_event",

            "graph_importance",

            "connectivity_level"

        ]

    ]

    .sort_values(

        "graph_importance",

        ascending=False

    )

)

,final_event,graph_importance,connectivity_level
18,Revenue Guidance,90.20,Very High
10,Layoffs,89.01,Very High
17,Product Launch,87.78,Very High
12,Market Movement,79.76,High
21,Technical Analysis,78.41,High
0,Analyst Rating,76.49,High
15,Other,74.51,High
4,Earnings,72.74,High
14,Options Activity,72.58,High
13,Merger & Acquisition,67.47,High


## 37. Save Updated Centrality

In [69]:
event_centrality.to_parquet(

    os.path.join(

        EVENT_PATH,

        "event_centrality.parquet"

    ),

    index=False

)

print("Saved Event Centrality")

Saved Event Centrality


## 38. Community Detection

use NetworkX's Greedy Modularity algorithm.

In [70]:
from networkx.algorithms.community import greedy_modularity_communities

communities = list(

    greedy_modularity_communities(

        G,

        weight="weight"

    )

)

print("Communities Found :", len(communities))

Communities Found : 4


## 39. Create Community Table

In [71]:
community_rows = []

for community_id, community in enumerate(communities):

    for event in community:

        community_rows.append(

            {

                "final_event": event,

                "community_id": community_id

            }

        )

event_communities = pd.DataFrame(

    community_rows

)

display(

    event_communities

    .sort_values(

        "community_id"

    )

)

,final_event,community_id
0,Insider Trading,0
1,Options Activity,0
2,Stock Split,0
3,Market Movement,0
4,Funding,0
5,Dividend,0
6,Merger & Acquisition,0
7,IPO,0
8,Share Buyback,0
9,Executive Change,0


## 40. Community Size

In [72]:
community_summary = (

    event_communities

    .groupby(

        "community_id"

    )

    .size()

    .reset_index(

        name="events"

    )

)

display(community_summary)

,community_id,events
0,0,14
1,1,12
2,2,3
3,3,1


## 41. Community Statistics

In [73]:
community_statistics = (

    event_communities

    .merge(

        event_statistics,

        on="final_event",

        how="left"

    )

)

display(

    community_statistics.head()

)

,final_event,community_id,total_news,companies,publishers,avg_confidence,bullish,bearish,neutral,signals,timeline_days,companies_score,publishers_score,confidence_score,timeline_score,influence_score,impact_category,event_fingerprint,explanation
0,Insider Trading,0,9530,2453,73,0.942794,118,17,9395,3,2011,0.246243,0.000000,0.949326,0.320513,37.16,Medium,Reliable,Companies: 2453 | Publishers: 73 | Confidence:...
1,Options Activity,0,43010,3651,190,0.860409,302,95,42603,4,2887,0.471262,0.163866,0.653886,0.632479,46.43,High,Widely Reported,Companies: 3651 | Publishers: 190 | Confidence...
2,Stock Split,0,6946,2490,121,0.760280,173,11,6761,4,1915,0.253193,0.067227,0.294815,0.286325,22.21,Low,,Companies: 2490 | Publishers: 121 | Confidence...
3,Market Movement,0,469086,6367,532,0.833923,0,0,469086,1,3816,0.981405,0.642857,0.558905,0.963319,78.84,Very High,"High Impact, Broad Coverage, Widely Reported",Companies: 6367 | Publishers: 532 | Confidence...
4,Funding,0,32578,4571,226,0.863721,3207,763,28445,4,2914,0.644065,0.214286,0.665762,0.642094,54.17,High,"Broad Coverage, Widely Reported",Companies: 4571 | Publishers: 226 | Confidence...


## 42. Automatically Name Communities

In [81]:
COMMUNITY_LABELS = {

    frozenset([
        "Earnings",
        "Revenue Guidance",
        "Analyst Rating"
    ]): "Corporate Performance",

    frozenset([
        "Merger & Acquisition",
        "Partnership",
        "Funding",
        "IPO"
    ]): "Corporate Growth",

    frozenset([
        "Fraud",
        "Bankruptcy",
        "Litigation"
    ]): "Legal & Financial Risk",

    frozenset([
        "Cybersecurity",
        "Cryptocurrency"
    ]): "Digital Assets & Security",

    frozenset([
        "Commodity",
        "Economic Data",
        "Monetary Policy"
    ]): "Macroeconomics",

    frozenset([
        "Dividend",
        "Share Buyback",
        "Stock Split"
    ]): "Capital Allocation"

}


def assign_community_name(events):

    event_set = set(events)

    for key, value in COMMUNITY_LABELS.items():

        if len(event_set & key) >= 2:

            return value

    return ", ".join(events[:3])


community_names = []

for cid, group in community_statistics.groupby("community_id"):

    events = (

        group

        .sort_values(

            "influence_score",

            ascending=False

        )["final_event"]

        .tolist()

    )

    community_names.append({

        "community_id": cid,

        "community_name": assign_community_name(events)

    })

community_names = pd.DataFrame(community_names)

display(community_names)

,community_id,community_name
0,0,Corporate Performance
1,1,Legal & Financial Risk
2,2,"Commodity, Revenue Guidance, ESG"
3,3,Cybersecurity


## 43. Merge Names

In [82]:
event_communities = (

    event_communities

    .merge(

        community_names,

        on="community_id"

    )

)

display(event_communities.head())

,final_event,community_id,community_name_x,community_name_y
0,Insider Trading,0,"Market Movement, Earnings, Analyst Rating",Corporate Performance
1,Options Activity,0,"Market Movement, Earnings, Analyst Rating",Corporate Performance
2,Stock Split,0,"Market Movement, Earnings, Analyst Rating",Corporate Performance
3,Market Movement,0,"Market Movement, Earnings, Analyst Rating",Corporate Performance
4,Funding,0,"Market Movement, Earnings, Analyst Rating",Corporate Performance


## 44. Save Communities

In [83]:
event_communities.to_parquet(

    os.path.join(

        EVENT_PATH,

        "event_communities.parquet"

    ),

    index=False

)

print("Saved Event Communities")

Saved Event Communities


## 45. Event Similarity Matrix

In [84]:
event_matrix = (

    financial_df

    .pivot_table(

        index="ticker",

        columns="final_event",

        values="news_id",

        aggfunc="count",

        fill_value=0

    )

)

print(event_matrix.shape)

(6619, 30)


## 46. Cosine Similarity

In [85]:
similarity = cosine_similarity(

    event_matrix.T

)

similarity = pd.DataFrame(

    similarity,

    index=event_matrix.columns,

    columns=event_matrix.columns

)

display(

    similarity.iloc[:5, :5]

)

final_event,Analyst Rating,Bankruptcy,Commodity,Credit Rating,Cryptocurrency
final_event,,,,,
Analyst Rating,1.000000,0.234433,0.294163,0.439871,0.215582
Bankruptcy,0.234433,1.000000,0.140134,0.140878,0.072553
Commodity,0.294163,0.140134,1.000000,0.189019,0.185245
Credit Rating,0.439871,0.140878,0.189019,1.000000,0.209010
Cryptocurrency,0.215582,0.072553,0.185245,0.209010,1.000000


## 47. Top Similar Events

In [86]:
rows = []

events = similarity.index.tolist()

for i in range(len(events)):

    for j in range(i + 1, len(events)):

        score = similarity.iloc[i, j]

        rows.append({

            "event_1": events[i],

            "event_2": events[j],

            "similarity": round(score, 4)

        })

event_similarity = (

    pd.DataFrame(rows)

    .sort_values(

        "similarity",

        ascending=False

    )

)

display(event_similarity.head(20))

,event_1,event_2,similarity
7,Analyst Rating,Earnings,0.8929
362,Market Movement,Product Launch,0.8721
212,Earnings,Market Movement,0.8194
418,Product Launch,Technical Analysis,0.8126
16,Analyst Rating,Market Movement,0.7919
218,Earnings,Product Launch,0.7888
367,Market Movement,Technical Analysis,0.7797
415,Product Launch,Revenue Guidance,0.7715
205,Earnings,Executive Change,0.7707
22,Analyst Rating,Product Launch,0.7706


## 48. Save Similarity

In [87]:
event_similarity.to_parquet(

    os.path.join(

        EVENT_PATH,

        "event_similarity.parquet"

    ),

    index=False

)

print("Saved Event Similarity")

Saved Event Similarity


## 49. Event Fingerprints

In [88]:
def create_event_fingerprint(row):

    fp = []

    if row["impact_category"] == "Very High":

        fp.append("High Impact")

    if row["graph_importance"] > 70:

        fp.append("Highly Connected")

    if row["avg_confidence"] > 0.90:

        fp.append("Reliable")

    if row["companies"] > event_statistics["companies"].median():

        fp.append("Broad Coverage")

    if row["publishers"] > event_statistics["publishers"].median():

        fp.append("Widely Reported")

    if row["bullish"] > row["bearish"]:

        fp.append("Bullish")

    elif row["bearish"] > row["bullish"]:

        fp.append("Bearish")

    return ", ".join(fp)

In [89]:
event_statistics = (

    event_statistics

    .merge(

        event_centrality[

            [

                "final_event",

                "graph_importance"

            ]

        ],

        on="final_event"

    )

)

event_statistics["event_fingerprint"] = (

    event_statistics.apply(

        create_event_fingerprint,

        axis=1

    )

)

display(

    event_statistics[

        [

            "final_event",

            "event_fingerprint"

        ]

    ]

)

,final_event,event_fingerprint
0,Analyst Rating,"High Impact, Highly Connected, Broad Coverage,..."
1,Bankruptcy,"Reliable, Bearish"
2,Commodity,"High Impact, Broad Coverage, Widely Reported, ..."
3,Credit Rating,"Reliable, Widely Reported, Bullish"
4,Cryptocurrency,Bullish
5,Cybersecurity,Bearish
6,Dividend,"High Impact, Reliable, Broad Coverage, Bullish"
7,ESG,"Broad Coverage, Widely Reported, Bullish"
8,Earnings,"High Impact, Highly Connected, Reliable, Broad..."
9,Economic Data,


## 51. Event Lifecycle

In [90]:
timeline = (

    financial_df

    .assign(

        year=pd.to_datetime(

            financial_df["published_date"]

        ).dt.year

    )

)

event_lifecycle = (

    timeline

    .groupby(

        [

            "final_event",

            "year"

        ]

    )

    .size()

    .reset_index(name="news_count")

)

In [91]:
life = []

for event, group in event_lifecycle.groupby("final_event"):

    peak = group.loc[

        group["news_count"].idxmax()

    ]

    life.append({

        "final_event": event,

        "first_year": group["year"].min(),

        "last_year": group["year"].max(),

        "peak_year": peak["year"],

        "peak_news": peak["news_count"]

    })

event_lifecycle_summary = pd.DataFrame(life)

display(event_lifecycle_summary)

,final_event,first_year,last_year,peak_year,peak_news
0,Analyst Rating,2009,2020,2013,34554
1,Bankruptcy,2009,2020,2016,623
2,Commodity,2009,2020,2019,32577
3,Credit Rating,2009,2020,2019,7785
4,Cryptocurrency,2009,2020,2018,2482
5,Cybersecurity,2009,2020,2015,1461
6,Dividend,2009,2020,2018,24999
7,ESG,2009,2020,2018,5507
8,Earnings,2009,2020,2019,100641
9,Economic Data,2009,2020,2018,2480


## 53. Event Heatmap

In [92]:
event_heatmap = (

    financial_df

    .assign(

        month=pd.to_datetime(

            financial_df["published_date"]

        ).dt.to_period("M").astype(str)

    )

    .pivot_table(

        index="month",

        columns="final_event",

        values="news_id",

        aggfunc="count",

        fill_value=0

    )

)

display(event_heatmap.head())

final_event,Analyst Rating,Bankruptcy,Commodity,Credit Rating,Cryptocurrency,Cybersecurity,Dividend,ESG,Earnings,Economic Data,...,Options Activity,Other,Partnership,Product Launch,Regulatory Approval,Revenue Guidance,Share Buyback,Stock Split,Technical Analysis,Trading Halt
month,,,,,,,,,,,,,,,,,,,,,
2009-02,0,0,0,0,1,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2009-04,0,0,0,0,0,0,0,0,0,0,...,0,0,0,3,0,0,0,0,0,0
2009-05,0,0,0,0,0,0,0,0,0,0,...,0,0,0,1,0,0,0,0,3,0
2009-06,0,0,0,0,0,0,0,0,4,0,...,0,5,0,10,0,0,0,0,2,0
2009-07,0,0,2,0,1,0,0,7,15,0,...,0,7,0,25,0,0,0,0,2,0


## 54. Event Knowledge Cards

In [93]:
knowledge_cards = (

    event_statistics

    .merge(

        event_lifecycle_summary,

        on="final_event"

    )

)

knowledge_cards["summary"] = (

    "Impact: "

    + knowledge_cards["impact_category"].astype(str)

    + " | Companies: "

    + knowledge_cards["companies"].astype(str)

    + " | Publishers: "

    + knowledge_cards["publishers"].astype(str)

    + " | Peak Year: "

    + knowledge_cards["peak_year"].astype(str)

)

display(

    knowledge_cards[

        [

            "final_event",

            "summary"

        ]

    ]

)

,final_event,summary
0,Analyst Rating,Impact: Very High | Companies: 5178 | Publishe...
1,Bankruptcy,Impact: Low | Companies: 1142 | Publishers: 93...
2,Commodity,Impact: Very High | Companies: 4965 | Publishe...
3,Credit Rating,Impact: High | Companies: 3955 | Publishers: 2...
4,Cryptocurrency,Impact: Low | Companies: 2556 | Publishers: 18...
5,Cybersecurity,Impact: Low | Companies: 1817 | Publishers: 17...
6,Dividend,Impact: Very High | Companies: 3981 | Publishe...
7,ESG,Impact: Medium | Companies: 4120 | Publishers:...
8,Earnings,Impact: Very High | Companies: 5694 | Publishe...
9,Economic Data,Impact: Low | Companies: 2222 | Publishers: 17...


## 55. Propagation Risk Score

In [94]:
risk_lookup = analytics_df.set_index(

    "ticker"

)["normalized_risk_score"]

company_event = (

    financial_df

    .merge(

        analytics_df[

            [

                "ticker",

                "normalized_risk_score"

            ]

        ],

        on="ticker"

    )

)

propagation_risk = (

    company_event

    .groupby("final_event")

    .agg(

        avg_company_risk=(

            "normalized_risk_score",

            "mean"

        )

    )

    .reset_index()

)

propagation_risk = propagation_risk.merge(

    event_statistics[

        [

            "final_event",

            "influence_score"

        ]

    ],

    on="final_event"

)

propagation_risk["propagation_risk"] = (

    propagation_risk["avg_company_risk"]

    *

    propagation_risk["influence_score"]

    /100

).round(2)

display(

    propagation_risk.sort_values(

        "propagation_risk",

        ascending=False

    )

)

,final_event,avg_company_risk,influence_score,propagation_risk
23,Product Launch,35.165064,82.23,28.92
17,Market Movement,35.353410,78.84,27.87
8,Earnings,33.076500,78.41,25.94
2,Commodity,36.530256,66.03,24.12
28,Technical Analysis,35.732690,65.30,23.33
18,Merger & Acquisition,34.250956,64.38,22.05
0,Analyst Rating,33.450122,65.78,22.00
3,Credit Rating,38.415356,57.10,21.94
10,Executive Change,34.831527,58.01,20.21
12,Funding,34.475569,54.17,18.68


## 56. Save

In [95]:
event_statistics.to_parquet(
    os.path.join(EVENT_PATH, "event_statistics.parquet"),
    index=False
)

event_communities.to_parquet(
    os.path.join(EVENT_PATH, "event_communities.parquet"),
    index=False
)

event_similarity.to_parquet(
    os.path.join(EVENT_PATH, "event_similarity.parquet"),
    index=False
)

event_lifecycle_summary.to_parquet(
    os.path.join(EVENT_PATH, "event_lifecycle.parquet"),
    index=False
)

event_heatmap.to_parquet(
    os.path.join(EVENT_PATH, "event_heatmap.parquet")
)

knowledge_cards.to_parquet(
    os.path.join(EVENT_PATH, "event_knowledge_cards.parquet"),
    index=False
)

propagation_risk.to_parquet(
    os.path.join(EVENT_PATH, "propagation_risk.parquet"),
    index=False
)

print("Event Propagation Analytics Saved Successfully")

Event Propagation Analytics Saved Successfully


## 57. Create Propagation Graph

In [108]:
import networkx as nx

propagation_graph = nx.Graph()

for row in event_edge_df.itertuples(index=False):

    propagation_graph.add_edge(

        row.event_a,

        row.event_b,

        weight=row.weight

    )

print("Nodes :", propagation_graph.number_of_nodes())
print("Edges :", propagation_graph.number_of_edges())

Nodes : 29
Edges : 221


In [109]:
for event in event_statistics["final_event"]:

    propagation_graph.add_node(event)

for row in event_edge_df.itertuples(index=False):

    propagation_graph.add_edge(

        row.event_a,

        row.event_b,

        weight=row.weight

    )

In [110]:
print("Nodes :", propagation_graph.number_of_nodes())
print("Edges :", propagation_graph.number_of_edges())

Nodes : 30
Edges : 221


## 58. Find Shortest Propagation Paths

In [111]:
propagation_paths = []

events = list(propagation_graph.nodes())

for source in events:

    for target in events:

        if source == target:
            continue

        try:

            path = nx.shortest_path(

                propagation_graph,

                source,

                target

            )

            propagation_paths.append({

                "source_event": source,

                "target_event": target,

                "path_length": len(path) - 1,

                "path": " --> ".join(path)

            })

        except nx.NetworkXNoPath:

            continue

propagation_paths = pd.DataFrame(propagation_paths)

print(propagation_paths.shape)

display(propagation_paths.head())

(812, 4)


,source_event,target_event,path_length,path
0,Analyst Rating,Credit Rating,1,Analyst Rating --> Credit Rating
1,Analyst Rating,Dividend,1,Analyst Rating --> Dividend
2,Analyst Rating,ESG,1,Analyst Rating --> ESG
3,Analyst Rating,Earnings,1,Analyst Rating --> Earnings
4,Analyst Rating,Executive Change,1,Analyst Rating --> Executive Change


## 59. Longest Interesting Paths

In [112]:
long_paths = (

    propagation_paths

    .sort_values(

        "path_length",

        ascending=False

    )

)

display(

    long_paths.head(20)

)

,source_event,target_event,path_length,path
647,Bankruptcy,ESG,3,Bankruptcy --> Layoffs --> Analyst Rating --> ESG
695,Commodity,Bankruptcy,3,Commodity --> Revenue Guidance --> Layoffs -->...
696,Commodity,Cryptocurrency,3,Commodity --> Revenue Guidance --> Economic Da...
699,Commodity,Regulatory Approval,3,Commodity --> Revenue Guidance --> Litigation ...
668,Bankruptcy,Cryptocurrency,3,Bankruptcy --> Layoffs --> Economic Data --> C...
671,Bankruptcy,Regulatory Approval,3,Bankruptcy --> Layoffs --> Litigation --> Regu...
667,Bankruptcy,Commodity,3,Bankruptcy --> Layoffs --> Revenue Guidance --...
807,Regulatory Approval,Bankruptcy,3,Regulatory Approval --> Litigation --> Layoffs...
274,Insider Trading,Bankruptcy,3,Insider Trading --> Analyst Rating --> Layoffs...
681,Commodity,Insider Trading,3,Commodity --> Revenue Guidance --> Analyst Rat...


## 60. Event Reachability

In [113]:
reachability = []

for event in propagation_graph.nodes():

    reachable = nx.single_source_shortest_path_length(

        propagation_graph,

        event

    )

    reachability.append({

        "final_event": event,

        "reachable_events": len(reachable) - 1,

        "average_distance": round(

            np.mean(list(reachable.values())),

            2

        )

    })

reachability = pd.DataFrame(reachability)

display(

    reachability

    .sort_values(

        "reachable_events",

        ascending=False

    )

)

,final_event,reachable_events,average_distance
0,Analyst Rating,28,1.17
1,Credit Rating,28,1.48
2,Dividend,28,1.52
3,ESG,28,1.79
4,Earnings,28,1.17
5,Executive Change,28,1.21
6,Fraud,28,1.52
7,Funding,28,1.38
8,IPO,28,1.38
9,Insider Trading,28,1.76


## 61. Event Bridges

In [114]:
bridge_score = pd.DataFrame({

    "final_event": list(

        nx.betweenness_centrality(

            propagation_graph,

            weight="weight"

        ).keys()

    ),

    "bridge_score": list(

        nx.betweenness_centrality(

            propagation_graph,

            weight="weight"

        ).values()

    )

})

bridge_score = bridge_score.sort_values(

    "bridge_score",

    ascending=False

)

display(bridge_score.head(20))

,final_event,bridge_score
10,Layoffs,0.088670
18,Revenue Guidance,0.076355
14,Options Activity,0.039409
15,Other,0.036946
22,Trading Halt,0.036946
21,Technical Analysis,0.034483
17,Product Launch,0.034483
16,Partnership,0.024631
0,Analyst Rating,0.024631
11,Litigation,0.019704


## 62. Event Hub Score

In [115]:
hub_scores = pd.DataFrame({

    "final_event": list(

        nx.degree_centrality(

            propagation_graph

        ).keys()

    ),

    "hub_score": list(

        nx.degree_centrality(

            propagation_graph

        ).values()

    )

})

display(

    hub_scores

    .sort_values(

        "hub_score",

        ascending=False

    )

)

,final_event,hub_score
17,Product Launch,0.896552
12,Market Movement,0.827586
18,Revenue Guidance,0.827586
21,Technical Analysis,0.793103
10,Layoffs,0.793103
0,Analyst Rating,0.758621
15,Other,0.758621
4,Earnings,0.758621
5,Executive Change,0.724138
13,Merger & Acquisition,0.724138


## 63. Merge Graph Analytics

In [116]:
event_graph_summary = (

    event_statistics

    .merge(

        bridge_score,

        on="final_event",

        how="left"

    )

    .merge(

        hub_scores,

        on="final_event",

        how="left"

    )

    .merge(

        reachability,

        on="final_event",

        how="left"

    )

)

display(event_graph_summary.head())

,final_event,total_news,companies,publishers,avg_confidence,bullish,bearish,neutral,signals,timeline_days,...,timeline_score,influence_score,impact_category,event_fingerprint,explanation,graph_importance,bridge_score,hub_score,reachable_events,average_distance
0,Analyst Rating,277248,5178,311,0.899277,78310,12691,184055,4,3185,...,0.738604,65.78,Very High,"High Impact, Highly Connected, Broad Coverage,...",Companies: 5178 | Publishers: 311 | Confidence...,76.49,0.024631,0.758621,28,1.17
1,Bankruptcy,3043,1142,93,0.904568,37,1826,1062,4,1111,...,0.000000,21.01,Low,"Reliable, Bearish",Companies: 1142 | Publishers: 93 | Confidence:...,12.01,0.000000,0.034483,28,2.07
2,Commodity,176547,4965,450,0.840405,9460,7778,157888,4,3572,...,0.876425,66.03,Very High,"High Impact, Broad Coverage, Widely Reported, ...",Companies: 4965 | Publishers: 450 | Confidence...,12.57,0.000000,0.034483,28,2.03
3,Credit Rating,36358,3955,208,0.956756,2697,1122,32488,4,2777,...,0.593305,57.10,High,"Reliable, Widely Reported, Bullish",Companies: 3955 | Publishers: 208 | Confidence...,42.90,0.000000,0.448276,28,1.48
4,Cryptocurrency,13739,2556,187,0.773372,119,74,13543,4,2379,...,0.451567,28.60,Low,Bullish,Companies: 2556 | Publishers: 187 | Confidence...,18.63,0.000000,0.137931,28,1.86


## 64. Save Propagation Analytics

In [117]:
propagation_paths.to_parquet(

    os.path.join(

        EVENT_PATH,

        "event_propagation_paths.parquet"

    ),

    index=False

)

event_graph_summary.to_parquet(

    os.path.join(

        EVENT_PATH,

        "event_graph_summary.parquet"

    ),

    index=False

)

print("Event Propagation Analytics Saved")

Event Propagation Analytics Saved


## 65. Summary

In [118]:
summary = pd.DataFrame({

    "Metric": [

        "Events",
        "Event Communities",
        "Event Similarity Pairs",
        "Propagation Paths",
        "Graph Nodes",
        "Graph Edges"

    ],

    "Value": [

        len(event_statistics),
        event_communities["community_id"].nunique(),
        len(event_similarity),
        len(propagation_paths),
        propagation_graph.number_of_nodes(),
        propagation_graph.number_of_edges()

    ]

})

display(summary)

,Metric,Value
0,Events,30
1,Event Communities,4
2,Event Similarity Pairs,435
3,Propagation Paths,812
4,Graph Nodes,30
5,Graph Edges,221
